# 06 多重积分与高维 Monte Carlo 框架

本 Notebook 建立多重积分和高维随机积分的基本框架。矩形区域上的张量积求积、变限积分和简单坐标变换已经给出可运行示例；一般不规则区域、自适应多重积分、拟 Monte Carlo 和方差缩减仍标注为后续待完善内容。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    composite_simpson_2d_rectangle,
    gauss_legendre_integrate,
    monte_carlo_integrate,
    rejection_monte_carlo_integrate,
    tensor_product_gauss_legendre,
    variable_limit_integral,
)


## 累次积分和张量积求积

矩形区域上的二维积分可以写成

$$
\int_a^b\int_c^d f(x,y)\,dy\,dx.
$$

若一维求积公式为

$$
\int_a^b g(x)\,dx\approx \sum_i w_i g(x_i),
$$

则二维张量积公式为

$$
\int_a^b\int_c^d f(x,y)\,dy\,dx
\approx
\sum_i\sum_j w_i v_j f(x_i,y_j).
$$


In [ ]:
def teaching_tensor_gauss(func, x_bounds, y_bounds, order):
    nodes, weights = np.polynomial.legendre.leggauss(order)
    ax, bx = x_bounds
    ay, by = y_bounds
    x = 0.5 * (ax + bx) + 0.5 * (bx - ax) * nodes
    y = 0.5 * (ay + by) + 0.5 * (by - ay) * nodes
    xx, yy = np.meshgrid(x, y, indexing="ij")
    scale = 0.25 * (bx - ax) * (by - ay)
    return scale * np.sum(weights[:, None] * weights[None, :] * func(xx, yy))

func = lambda x, y: x + y
print("教学版张量积 Gauss:", teaching_tensor_gauss(func, (0.0, 1.0), (0.0, 2.0), 4))
print("正式实现张量积 Gauss:", tensor_product_gauss_legendre(func, (0.0, 1.0), (0.0, 2.0), 4, 4))
print("精确值:", 3.0)


## 矩形区域上的复合 Simpson

对二维矩形区域，可以把一维复合 Simpson 权重做张量积。若两个方向的子区间数分别为 $n_x,n_y$，且都为偶数，则节点数为

$$
(n_x+1)(n_y+1).
$$

这已经显示出维数增加时的成本增长。


In [ ]:
value_gauss = tensor_product_gauss_legendre(lambda x, y: np.exp(-(x**2 + y**2)), (-1.0, 1.0), (-1.0, 1.0), 16, 16)
value_simpson = composite_simpson_2d_rectangle(lambda x, y: np.exp(-(x**2 + y**2)), (-1.0, 1.0), (-1.0, 1.0), 40, 40)
print("二维 Gauss-Legendre:", value_gauss)
print("二维复合 Simpson:", value_simpson)
print("二者差值:", abs(value_gauss - value_simpson))


## 一般区域的变限积分

简单非矩形区域可以写成变限积分。例如三角形

$$
\Omega=\{(x,y):0\le x\le 1,\;0\le y\le 1-x\}
$$

的面积是

$$
\int_0^1\int_0^{1-x}1\,dy\,dx=\frac12.
$$


In [ ]:
triangle_area = variable_limit_integral(
    lambda x, y: np.ones_like(x),
    (0.0, 1.0),
    lambda x: np.zeros_like(x),
    lambda x: 1.0 - x,
    x_order=16,
    y_order=16,
)
print("三角形面积:", triangle_area)


## 坐标变换与 Jacobian

对单位圆盘，极坐标变换为

$$
x=r\cos\theta,\qquad y=r\sin\theta,
$$

Jacobian 为 $r$。因此面积积分为

$$
\int_0^{2\pi}\int_0^1 r\,dr\,d\theta=\pi.
$$


In [ ]:
disk_area = tensor_product_gauss_legendre(lambda r, theta: r, (0.0, 1.0), (0.0, 2 * math.pi), 20, 20)
print("极坐标积分得到的单位圆面积:", disk_area)
print("误差:", abs(disk_area - math.pi))


## 维数灾难

张量积求积在低维很有效，但维数增加时节点数按指数增长。如果每个方向使用 $m$ 个节点，$d$ 维中需要 $m^d$ 个函数值。


In [ ]:
per_dim = 8
dims = np.arange(1, 11)
node_counts = per_dim**dims

plt.figure(figsize=(7, 4.0))
plt.semilogy(dims, node_counts, "o-")
plt.xlabel("维数 d")
plt.ylabel(f"节点数 {per_dim}^d")
plt.title("张量积求积的节点数随维数指数增长")
plt.grid(True, which="both", alpha=0.3)
plt.show()


## Monte Carlo 积分

若 $X$ 在区域 $\Omega$ 上均匀分布，体积为 $|\Omega|$，则

$$
\int_\Omega f(x)\,dx=|\Omega|\,\mathbb E[f(X)].
$$

Monte Carlo 估计量为

$$
\hat I_N=|\Omega|\frac1N\sum_{i=1}^N f(X_i),
$$

标准误差近似为

$$
|\Omega|\sqrt{\frac{s^2}{N}}.
$$

它的典型误差阶是 $O(N^{-1/2})$，不如低维高阶求积快，但对维数没有显式指数依赖。


In [ ]:
dimension = 6
bounds = np.array([[0.0, 1.0]] * dimension)
one_dim = gauss_legendre_integrate(lambda x: np.exp(-x**2), 0.0, 1.0, 64)
reference = one_dim**dimension
sample_sizes = np.array([500, 1_000, 2_000, 5_000, 10_000, 20_000])
results = []
for n in sample_sizes:
    result = monte_carlo_integrate(lambda points: np.exp(-np.sum(points**2, axis=1)), bounds, int(n), seed=2026)
    results.append((result.value, result.standard_error))
results = np.array(results)

print("N      估计值        标准误差       与参考值差")
for n, (value, se) in zip(sample_sizes, results):
    print(f"{n:<6d} {value: .8f}  {se: .3e}  {abs(value-reference): .3e}")

plt.figure(figsize=(7, 4.0))
plt.loglog(sample_sizes, np.abs(results[:, 0] - reference), "o-", label="实际误差")
plt.loglog(sample_sizes, results[:, 1], "s-", label="标准误差估计")
plt.loglog(sample_sizes, results[0, 1] * np.sqrt(sample_sizes[0] / sample_sizes), "--", color="gray", label="$N^{-1/2}$ 参考")
plt.xlabel("样本量 N")
plt.ylabel("误差或标准误差")
plt.title("高维 Monte Carlo 积分收敛实验")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 待完善内容和小结

本 Notebook 已建立：累次积分、张量积求积、矩形区域复合 Simpson、变限积分、坐标变换、Jacobian、维数灾难和 Monte Carlo 估计。

以下内容仍是后续待完善框架：

* 不规则区域的系统网格剖分和自适应多重积分；
* 拒绝采样效率分析；
* 重要性采样、控制变量和分层采样等方差缩减；
* Sobol、Halton 等拟 Monte Carlo 方法；
* 奇异区域和高维物理积分的专题案例。
